# Notebook 17 — Plasticity

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 13 examined **stability**:

> Does useful structure persist across context changes?

Notebook 17 examines **plasticity**:

> Can the system incorporate useful new structure after context changes?

Together:

\[
\text{continual learning}
=
\text{stability}
+
\text{plasticity}
\]

Outputs:

- `results/17_boundary_recovery.csv`
- `results/17_plasticity_by_variant.csv`
- `results/17_recovery_trajectory.csv`
- `results/17_plasticity_states.csv`
- `results/17_plasticity_summary.json`
- `figures/17_boundary_recovery.png`
- `figures/17_plasticity_by_variant.png`
- `figures/17_recovery_trajectory.png`
- `figures/17_stability_plasticity_plane.png`
- `figures/17_plasticity_states.png`
- `results/17_plasticity_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain
from src.stability import compute_stability, compute_plasticity

print("Imports complete.")

## 2. Load Prior Trajectory

Notebook 17 prefers the context-stability output from Notebook 13:

```text
results/13_context_stability.csv
```

Fallback order:

1. `results/13_context_stability.csv`
2. `results/11_state_vs_stateless_instances.csv`
3. `results/01_gain_signal_trajectory.csv`
4. `results/00_context_gain.csv`
5. synthetic toy trajectory

In [ ]:
trajectory_13 = RESULTS_DIR / "13_context_stability.csv"
trajectory_11 = RESULTS_DIR / "11_state_vs_stateless_instances.csv"
trajectory_01 = RESULTS_DIR / "01_gain_signal_trajectory.csv"
trajectory_00 = RESULTS_DIR / "00_context_gain.csv"

if trajectory_13.exists():
    df = pd.read_csv(trajectory_13)
    source_file = trajectory_13
    print("Loaded:", trajectory_13)
elif trajectory_11.exists():
    df = pd.read_csv(trajectory_11)
    source_file = trajectory_11
    print("Loaded:", trajectory_11)
elif trajectory_01.exists():
    df = pd.read_csv(trajectory_01)
    source_file = trajectory_01
    print("Loaded:", trajectory_01)
elif trajectory_00.exists():
    df = pd.read_csv(trajectory_00)
    source_file = trajectory_00
    print("Loaded:", trajectory_00)
else:
    print("No prior trajectory found. Creating fallback toy trajectory.")
    df = pd.DataFrame({
        "instance": np.arange(1, 13),
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })
    source_file = None

if "gain" not in df.columns:
    if "state_advantage" in df.columns:
        df["gain"] = df["state_advantage"]
    else:
        df["gain"] = compute_gain(
            df["stateful_reward"].tolist(),
            df["stateless_reward"].tolist(),
        )

if "gain_delta" not in df.columns:
    df["gain_delta"] = df["gain"].diff().fillna(0.0)

if "cumulative_gain" not in df.columns:
    df["cumulative_gain"] = df["gain"].cumsum()

df = df.sort_values("instance").reset_index(drop=True)
df

## 3. Define Plasticity

Plasticity is the ability to improve after new evidence or a context transition.

Operationally, this notebook measures:

\[
\text{recovery}
=
\max(g_{after})
-
g_{boundary}
\]

where:

- \(g_{boundary}\) is gain at the first instance of a new variant
- \(\max(g_{after})\) is the strongest gain reached later in that variant

Interpretation:

- positive recovery → adaptation
- zero recovery → rigidity
- negative recovery → collapse

## 4. Variant Boundaries

In [ ]:
variant_order = (
    df[["variant", "instance"]]
    .drop_duplicates("variant")
    .sort_values("instance")
)

boundary_instances = variant_order["instance"].tolist()
boundary_indices = [
    int(df.index[df["instance"] == inst][0])
    for inst in boundary_instances
]

boundary_table = pd.DataFrame({
    "variant": variant_order["variant"].tolist(),
    "boundary_instance": boundary_instances,
    "zero_based_index": boundary_indices,
})

boundary_table

## 5. Boundary Recovery Analysis

For each variant, measure whether gain recovers after the boundary.

In [ ]:
recovery_rows = []

for _, boundary in boundary_table.iterrows():
    variant = boundary["variant"]
    boundary_instance = int(boundary["boundary_instance"])

    group = df[df["variant"] == variant].copy()
    group = group.sort_values("instance")

    boundary_gain = float(group["gain"].iloc[0])
    peak_gain = float(group["gain"].max())
    final_gain = float(group["gain"].iloc[-1])
    recovery_gain = peak_gain - boundary_gain
    final_recovery = final_gain - boundary_gain

    if recovery_gain > 0.08:
        recovery_label = "adaptive"
    elif recovery_gain > 0:
        recovery_label = "gradual"
    elif recovery_gain == 0:
        recovery_label = "rigid"
    else:
        recovery_label = "collapsing"

    recovery_rows.append({
        "variant": variant,
        "boundary_instance": boundary_instance,
        "boundary_gain": boundary_gain,
        "peak_gain_after_boundary": peak_gain,
        "final_gain": final_gain,
        "recovery_gain": float(recovery_gain),
        "final_recovery": float(final_recovery),
        "plasticity_label": recovery_label,
    })

boundary_recovery = pd.DataFrame(recovery_rows)
boundary_recovery

## 6. Plasticity by Variant

This section measures positive within-variant gain updates.

The core signal:

```text
mean positive gain delta
```

Positive gain deltas indicate adaptation.

In [ ]:
plasticity_rows = []

for variant, group in df.groupby("variant", sort=False):
    group = group.sort_values("instance").copy()
    positive_deltas = group.loc[group["gain_delta"] > 0, "gain_delta"]

    mean_positive_delta = (
        float(positive_deltas.mean())
        if len(positive_deltas) > 0
        else 0.0
    )

    total_positive_delta = float(positive_deltas.sum())

    plasticity_rows.append({
        "variant": variant,
        "instances": int(len(group)),
        "mean_gain": float(group["gain"].mean()),
        "mean_gain_delta": float(group["gain_delta"].mean()),
        "mean_positive_gain_delta": mean_positive_delta,
        "total_positive_gain_delta": total_positive_delta,
        "positive_delta_count": int((group["gain_delta"] > 0).sum()),
        "negative_delta_count": int((group["gain_delta"] < 0).sum()),
    })

plasticity_by_variant = pd.DataFrame(plasticity_rows)
plasticity_by_variant

## 7. Recovery Trajectory

Measure average gain and gain change by distance from a variant boundary.

In [ ]:
recovery_df = df.copy()

distance_values = []

for _, row in recovery_df.iterrows():
    variant = row["variant"]
    inst = int(row["instance"])
    boundary_inst = int(
        boundary_table.loc[
            boundary_table["variant"] == variant,
            "boundary_instance"
        ].iloc[0]
    )
    distance_values.append(inst - boundary_inst)

recovery_df["distance_from_variant_boundary"] = distance_values

recovery_trajectory = (
    recovery_df.groupby("distance_from_variant_boundary")
    .agg(
        mean_gain=("gain", "mean"),
        mean_gain_delta=("gain_delta", "mean"),
        instances=("instance", "count"),
    )
    .reset_index()
)

recovery_trajectory

## 8. Stability–Plasticity Plane

This is the central figure for Notebook 17.

For each variant:

- x-axis: stability signal, approximated by boundary gain
- y-axis: plasticity signal, approximated by recovery gain

Quadrants:

- high stability + high plasticity → continual learner
- high stability + low plasticity → rigid expert
- low stability + high plasticity → adaptive but initially fragile
- low stability + low plasticity → weak learning

In [ ]:
plane_df = boundary_recovery.copy()
plane_df = plane_df.rename(columns={
    "boundary_gain": "stability_signal",
    "recovery_gain": "plasticity_signal",
})

stability_threshold = float(plane_df["stability_signal"].median())
plasticity_threshold = float(plane_df["plasticity_signal"].median())

def classify_quadrant(row):
    stable = row["stability_signal"] >= stability_threshold
    plastic = row["plasticity_signal"] >= plasticity_threshold

    if stable and plastic:
        return "continual_learner"
    if stable and not plastic:
        return "rigid_expert"
    if not stable and plastic:
        return "adaptive_fragile"
    return "weak_learning"

plane_df["stability_plasticity_quadrant"] = plane_df.apply(classify_quadrant, axis=1)

plane_df

## 9. Plasticity States

Assign an instance-level plasticity state:

- **adaptive**: positive gain change
- **holding**: gain positive and unchanged
- **rigid**: zero gain
- **declining**: gain positive but decreasing
- **collapsing**: negative gain

In [ ]:
state_labels = []

for _, row in df.iterrows():
    gain = float(row["gain"])
    delta = float(row["gain_delta"])

    if gain < 0:
        label = "collapsing"
    elif delta > 0:
        label = "adaptive"
    elif gain > 0 and delta == 0:
        label = "holding"
    elif gain == 0:
        label = "rigid"
    elif gain > 0 and delta < 0:
        label = "declining"
    else:
        label = "unknown"

    state_labels.append(label)

plasticity_states = df.copy()
plasticity_states["plasticity_state"] = state_labels

plasticity_states[[
    "instance",
    "variant",
    "gain",
    "gain_delta",
    "plasticity_state",
]]

## 10. Figure — Boundary Recovery

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    boundary_recovery["variant"],
    boundary_recovery["recovery_gain"],
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 17: Boundary Recovery")
ax.set_xlabel("Variant")
ax.set_ylabel("Peak Gain After Boundary - Boundary Gain")
ax.grid(True, axis="y", alpha=0.3)

boundary_recovery_path = FIGURES_DIR / "17_boundary_recovery.png"
fig.tight_layout()
fig.savefig(boundary_recovery_path, dpi=160)

boundary_recovery_path

## 11. Figure — Plasticity by Variant

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    plasticity_by_variant["variant"],
    plasticity_by_variant["mean_positive_gain_delta"],
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 17: Mean Positive Gain Delta by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Mean Positive Gain Delta")
ax.grid(True, axis="y", alpha=0.3)

plasticity_variant_path = FIGURES_DIR / "17_plasticity_by_variant.png"
fig.tight_layout()
fig.savefig(plasticity_variant_path, dpi=160)

plasticity_variant_path

## 12. Figure — Recovery Trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    recovery_trajectory["distance_from_variant_boundary"],
    recovery_trajectory["mean_gain"],
    marker="o",
    label="Mean gain",
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 17: Recovery Trajectory")
ax.set_xlabel("Distance From Variant Boundary")
ax.set_ylabel("Mean Gain")
ax.legend()
ax.grid(True, alpha=0.3)

recovery_trajectory_path = FIGURES_DIR / "17_recovery_trajectory.png"
fig.tight_layout()
fig.savefig(recovery_trajectory_path, dpi=160)

recovery_trajectory_path

## 13. Figure — Stability–Plasticity Plane

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    plane_df["stability_signal"],
    plane_df["plasticity_signal"],
    s=120,
)

for _, row in plane_df.iterrows():
    ax.annotate(
        row["variant"],
        (row["stability_signal"], row["plasticity_signal"]),
        textcoords="offset points",
        xytext=(8, 8),
    )

ax.axvline(stability_threshold, linestyle="--", linewidth=1)
ax.axhline(plasticity_threshold, linestyle="--", linewidth=1)

ax.set_title("Notebook 17: Stability–Plasticity Plane")
ax.set_xlabel("Stability Signal (Boundary Gain)")
ax.set_ylabel("Plasticity Signal (Recovery Gain)")
ax.grid(True, alpha=0.3)

stability_plasticity_path = FIGURES_DIR / "17_stability_plasticity_plane.png"
fig.tight_layout()
fig.savefig(stability_plasticity_path, dpi=160)

stability_plasticity_path

## 14. Figure — Plasticity States

In [ ]:
state_counts = (
    plasticity_states
    .groupby(["variant", "plasticity_state"])
    .size()
    .unstack(fill_value=0)
)

preferred_order = ["adaptive", "holding", "rigid", "declining", "collapsing"]
for col in preferred_order:
    if col not in state_counts.columns:
        state_counts[col] = 0

state_counts = state_counts[preferred_order]

fig, ax = plt.subplots(figsize=(9, 5))

bottom = np.zeros(len(state_counts))

for column in state_counts.columns:
    values = state_counts[column].values
    if values.sum() == 0:
        continue
    ax.bar(state_counts.index, values, bottom=bottom, label=column)
    bottom += values

ax.set_title("Notebook 17: Plasticity States by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Instance Count")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

plasticity_states_path = FIGURES_DIR / "17_plasticity_states.png"
fig.tight_layout()
fig.savefig(plasticity_states_path, dpi=160)

plasticity_states_path

## 15. Summary

In [ ]:
variant_boundaries = [
    int(df.index[df["instance"] == inst][0])
    for inst in boundary_instances
]

basic_stability = compute_stability(
    df["gain"].tolist(),
    variant_boundaries,
)

basic_plasticity = compute_plasticity(
    df["gain"].tolist(),
    variant_boundaries,
)

summary = {
    "total_instances": int(len(df)),
    "variants": df["variant"].drop_duplicates().tolist(),
    "trajectory_source": str(source_file) if source_file is not None else "fallback",
    "mean_gain": float(df["gain"].mean()),
    "total_gain": float(df["gain"].sum()),
    "basic_stability_boundary_gain": float(basic_stability),
    "basic_plasticity_non_boundary_gain": float(basic_plasticity),
    "mean_recovery_gain": float(boundary_recovery["recovery_gain"].mean()),
    "max_recovery_gain": float(boundary_recovery["recovery_gain"].max()),
    "max_recovery_variant": str(
        boundary_recovery.loc[
            boundary_recovery["recovery_gain"].idxmax(),
            "variant"
        ]
    ),
    "stability_threshold": stability_threshold,
    "plasticity_threshold": plasticity_threshold,
    "quadrant_counts": {
        q: int(c)
        for q, c in plane_df["stability_plasticity_quadrant"].value_counts().items()
    },
    "plasticity_state_counts": {
        state: int(count)
        for state, count in plasticity_states["plasticity_state"].value_counts().items()
    },
}

summary

## 16. Export Artifacts

In [ ]:
boundary_recovery_csv = RESULTS_DIR / "17_boundary_recovery.csv"
plasticity_by_variant_csv = RESULTS_DIR / "17_plasticity_by_variant.csv"
recovery_trajectory_csv = RESULTS_DIR / "17_recovery_trajectory.csv"
plane_csv = RESULTS_DIR / "17_stability_plasticity_plane.csv"
plasticity_states_csv = RESULTS_DIR / "17_plasticity_states.csv"
summary_path = RESULTS_DIR / "17_plasticity_summary.json"
zip_path = RESULTS_DIR / "17_plasticity_artifacts.zip"

boundary_recovery.to_csv(boundary_recovery_csv, index=False)
plasticity_by_variant.to_csv(plasticity_by_variant_csv, index=False)
recovery_trajectory.to_csv(recovery_trajectory_csv, index=False)
plane_df.to_csv(plane_csv, index=False)
plasticity_states.to_csv(plasticity_states_csv, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "17_plasticity.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(summary_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        boundary_recovery_csv,
        plasticity_by_variant_csv,
        recovery_trajectory_csv,
        plane_csv,
        plasticity_states_csv,
        summary_path,
        boundary_recovery_path,
        plasticity_variant_path,
        recovery_trajectory_path,
        stability_plasticity_path,
        plasticity_states_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    boundary_recovery_csv,
    plasticity_by_variant_csv,
    recovery_trajectory_csv,
    plane_csv,
    plasticity_states_csv,
    summary_path,
    boundary_recovery_path,
    plasticity_variant_path,
    recovery_trajectory_path,
    stability_plasticity_path,
    plasticity_states_path,
    zip_path,
]:
    print("-", path)

## 17. Download Artifacts

In Colab, this downloads the zip.

Locally, the archive is saved under:

```text
rml/results/17_plasticity_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 18. Notebook 17 Claim

Stability preserves useful structure.

Plasticity incorporates useful new structure.

Continual learning requires both:

\[
\text{continual learning}
=
\text{stability}
+
\text{plasticity}
\]

In RML terms:

> A system must retain what remains useful while adapting to what changes.

Notebook 19 will next examine stale context:

> When does retained structure become harmful?